# Продвинутые техники

## Декораторы для функции с любыми аргументами

### Шаг 1: Стлакиваемся с проблемой: наши первые декораторы не универсальны

В предыдущем модуле мы создали несколько очень полезных декораторов: @timer, @logger и @cache. Вы могли заметить, что в декораторе @loger мы уже использовали *args и **kwargs, но мы не акцентировали на этом внимание. Давайте сейчас разберемся, почему без них наши декораторы были бы крайне ограниченными.

Вспомним самый первый декоратор, который мы написали в Модуле 1. Его структура была примерно такой:

In [2]:
def simple_wrapper(func):
    # Обратите внимание на сигнатуру обертки!
    def wrapper(): 
        print("Вход в функцию")
        func() # И здесь вызов без аргументов
        print("Выход из функции")
    return wrapper

Этот декоратор отлично работает для функций, которые не принимают никаких аргументов, например:

In [4]:
@simple_wrapper
def say_hello():
    print("Привет!")

say_hello() # Это сработает без проблем

Вход в функцию
Привет!
Выход из функции


#### А теперь давайте сломаем его!

Попробуем применить этот же, казалось бы, рабочий декоратор к функции, которая принимает аргументы. Например, к простому сумматору:

In [ ]:
@simple_wrapper
def add(a, b):
    print(f"Сумма: {a + b}")

# А теперь попробуем вызвать нашу "украшенную" функцию
try:
    add(5, 10)
except TypeError as e:
    print(f"Произошла ошибка: {e}")

Произошла ошибка: simple_wrapper.<locals>.wrapper() takes 0 positional arguments but 2 were given


Запустив этот код, мы немедленно получим крах программы с ошибкой.

#### Почему это произшло? Давайте разберем по шагам:

1. Когда мы декорировали add с помощью @simple_wrapper, переменная add стала ссылаться не на нашу исходную функцию, а на вложенную функцию wrapper.

2. Далее мы вызываем add(5, 10).

3. На самом деле, Python пытается выполнить wrapper(5, 10).

4. Но наша функция wrapper определена как def wrapper(): - она не умеет принимать никаких аргументов!

5. Python видит это несоотвтествие и справедливо выбрасывает исключение TypeError, сообщая, что мы передали 2 аргумента функции, которая не ожидала не одного.

#### Формулируем проблему

Проблема очевидна: наши простые декораторы <b>жестко привязаны к сигнатуре</b> (списку аргументов) декорируемой функции. Декоратор, написанный для функции без аргументов сломается на функции с одним аргументом. Декоратор для функции с одним аргументом слоамается на функции с двумя, и так далее.

Это делает их практически бесполезными в реальных проектах, где функции могут иметь самые разные наборы параметров.

#### Итог

Базовый шаблон декоратора, который мы использовали вначале, хрупок и не универсален. Он ломается, как только мы пытаемся применить его к функции, принимающей аргументы. Эта проблема - основная причина, по которой нам необходимо освоить более мощные инструменты: *args и **kwargs. Именно они позволят нам создавать декораторы, которые будут работать с <b>любыми</b> фукнциями.

### Шаг 2: Вводим конструкции *args и **kwargs и объясняем их роль

Чтобы решить проблему из предыдущего шага, нам нужен способ создать функцию-обертку, которая была бы "универсальным солдатом" - способный принять <b>любое количество</b> аргументов <b>любого типа</b> (позиционных и именованных) и передать их дальше без изменений.

Именно для этой цели в Python существуют две специальные синтаксические конструкции: *args и **kwargs.

#### 1. *args для позиционных аргументов

Символ * перед именем параметра в определении функции говорит Python: "Собери все оставшиеся <b>позиционные</b> аргументы в <b>кортеж (tuple)</b> и присвой его этому параметру".

Давайте посмотрим на примере:

In [6]:
def demo_args(*args):
    print(f"Тип args: {type(args)}")
    print(f"Содержимое args: {args}")

print("Вызов с 0 аргументами:")
demo_args()

print("\nВызов с 1 аргументом:")
demo_args("hello")

print("\nВызов с 3 аргументами:")
demo_args(1, True, "world")

Вызов с 0 аргументами:
Тип args: <class 'tuple'>
Содержимое args: ()

Вызов с 1 аргументом:
Тип args: <class 'tuple'>
Содержимое args: ('hello',)

Вызов с 3 аргументами:
Тип args: <class 'tuple'>
Содержимое args: (1, True, 'world')


Как видите, args - это всегда кортеж, который содержит все позиционные аргументы, которые мы передали.

#### 2. **kwargs для именованных аргументов

Символ ** перед именем параметра работает похожим образом, но для именованных (keyword) аргументов. Он говорит Python: "Собери все <b>именованные</b> аргументы (ключ=значение) в <b>словарь (dict)</b> и присвой его этому параметру".

Пример:

In [7]:
def demo_kwargs(**kwargs):
    print(f"Тип kwargs: {type(kwargs)}")
    print(f"Содержимое kwargs: {kwargs}")

print("Вызов с 0 аргументами:")
demo_kwargs()

print("\nВызов с 1 именованным аргументом:")
demo_kwargs(name="Алиса")

print("\nВызов с 3 именованными аргументами:")
demo_kwargs(user_id=101, status="active", country="RU")

Вызов с 0 аргументами:
Тип kwargs: <class 'dict'>
Содержимое kwargs: {}

Вызов с 1 именованным аргументом:
Тип kwargs: <class 'dict'>
Содержимое kwargs: {'name': 'Алиса'}

Вызов с 3 именованными аргументами:
Тип kwargs: <class 'dict'>
Содержимое kwargs: {'user_id': 101, 'status': 'active', 'country': 'RU'}


kwargs - это всегда словарь, содержащий все переданные именованные аргументы.

#### 3. Совместное использование

Мы можем использовать обе конструкции в одной функции, чтобы принимать любую комбинацию аргументов. <b>Важное правило: *args всегда должен идти перед **kwargs</b>.

In [9]:
def universal_acceptor(*args, **kwargs):
    print("--- Получены аргументы ---")
    print(f"Позиционные (args): {args}")
    print(f"Именованные (kwargs): {kwargs}")

universal_acceptor(1, 2, 3, name="Боб", age=30)

--- Получены аргументы ---
Позиционные (args): (1, 2, 3)
Именованные (kwargs): {'name': 'Боб', 'age': 30}


#### Как это решает нашу проблему с декораторами?

Теперь мы можем создать идеальную, универсальную функцию-обертку!

1. Мы определяем ее как def wrapper(*args, **kwargs). Теперь она может <b>принять</b> абсолютно любые аргументы, которые предназначались для исходной функции.

2. Внутри wrapper мы вызываем исходную функцию func, <b>передавая ("пробрасывая")</b> ей эти собранные аргументы в том же виде: func(*args, **kwargs).

Синтаксис * и ** работает в обе стороны:

- В определении функции (def wrapper(*args, **kwrags)) он <b>собирает</b> аргументы в кортеж и словарь.

- В вызове функции (func(*args, **kwargs)) он <b>распаковывает</b> кортеж и словарь обратно в аргументы.

#### Итог

Конструкции *args и **kwargs - это стандартный механизм в Python для создания функций, способных работать с произвольным набором аргументов. Для декораторов это не просто удобство, а <b>необходимость</b>, позволяющая создаавать по-настоящему универсальные и переиспользуемые инструменты. Любой профессиональный декоратор должен использовать *args и **kwargs в своей функции-обертке.

### Шаг 3: Модифицируем внутреннюю функцию-обертку, чтобы она принимала *args, **kwargs

Наша задача - взять шаблон нашего старого. неуниверсального декоратора и превратить его в мощный инструмент, способный "обернуть" любую функцию.

Давайте вспомним, как выглядел наш проблемный декоратор:

In [10]:
# "ДО": Неуниверсальная версия
def simple_wrapper(func):
    def wrapper(): # <--- Проблема №1: не принимает аргументы
        print("--- Логика до вызова ---")
        func()     # <--- Проблема №2: вызывает без аргументов
        print("--- Логика после вызова ---")
    return wrapper

Теперь давайте исправим обе эти проблемы, применив наши новые знания.

#### "ПОСЛЕ": Универсальная версия декоратора

In [11]:
def universal_decorator(func):
    """
    Универсальный декоратор, который работает с любой функцией.
    """
    # 1. Определяем обертку, которая ПРИНИМАЕТ любые аргументы
    def wrapper(*args, **kwargs):
        
        print(f"--- Логика ДО вызова функции {func.__name__} ---")
        print(f"    Получены позиционные аргументы: {args}")
        print(f"    Получены именованные аргументы: {kwargs}")
        
        # 2. Вызываем исходную функцию, ПЕРЕДАВАЯ ей все аргументы
        result = func(*args, **kwargs)
        
        print(f"--- Логика ПОСЛЕ вызова функции {func.__name__} ---")
        print(f"    Функция вернула результат: {result}")
        
        # 3. Не забываем вернуть результат исходной функции
        return result
        
    return wrapper

<b>Ключевые изменения</b>:

1. <b>Сигнатура обертки</b>: def wrapper() превратилась в def wrapper(*args, **kwargs). Теперь наша wrapper может принять любое количество позиционных и именованных аругментов. Все позиционные аргументы "упакуются" в кортеж args, а именованные - в словарь kwargs.

2. <b>Вызов исходной функции</b>: func() превратился в func(*args, **kwargs). Здесь происходит обратный процесс - "распаковка". Символ * перед args говорит Python: "Возьми этот кортеж и распакуй его в позиционные аргументы для func". Аналогично, **kwargs говрит: "Возьми этот словарь и распакуй его в именованные аргументы для func".

#### Демонстрация работы

Давайте применим наш новый универсальный декоратор к нескольким совершенно разным функциям и убедимся, что он справляется со всеми.

In [12]:
@universal_decorator
def add(a, b):
    """Складывает два числа."""
    return a + b

@universal_decorator
def greet(name, title="Mr."):
    """Формирует приветствие."""
    return f"Hello, {title} {name}!"

@universal_decorator
def do_nothing():
    """Ничего не делает."""
    print("    (Выполняется тело функции do_nothing)")

# --- Тестируем ---
print("--- Вызов add(5, 10) ---")
add(5, 10)

print("\n" + "="*40 + "\n")

print("--- Вызов greet('Smith', title='Dr.') ---")
greet(name="Smith", title="Dr.")

print("\n" + "="*40 + "\n")

print("--- Вызов do_nothing() ---")
do_nothing()

--- Вызов add(5, 10) ---
--- Логика ДО вызова функции add ---
    Получены позиционные аргументы: (5, 10)
    Получены именованные аргументы: {}
--- Логика ПОСЛЕ вызова функции add ---
    Функция вернула результат: 15


--- Вызов greet('Smith', title='Dr.') ---
--- Логика ДО вызова функции greet ---
    Получены позиционные аргументы: ()
    Получены именованные аргументы: {'name': 'Smith', 'title': 'Dr.'}
--- Логика ПОСЛЕ вызова функции greet ---
    Функция вернула результат: Hello, Dr. Smith!


--- Вызов do_nothing() ---
--- Логика ДО вызова функции do_nothing ---
    Получены позиционные аргументы: ()
    Получены именованные аргументы: {}
    (Выполняется тело функции do_nothing)
--- Логика ПОСЛЕ вызова функции do_nothing ---
    Функция вернула результат: None


#### Анализ результата

Наш декоратор успешно справился со всеми тремя случаями:
- Для add(5, 10) он правильно "поймал" args = (5, 10) и kwargs = {}.
- Для greet(name='Smith', title='Dr.') он поймал args = () и kwargs = {'name': 'Smith', 'title': 'Dr.'}.
- Для do_nothing() он поймал args = () и kwargs = {}.

Во всех случаях аргументы были корректно "проброшены" в исходную функцию, и ее результат был так же корректно перехвачен и возвращен.

#### Итог

Шаблон def wrapper(\*args, *\*kwargs): ... result = func(\*args, **kwargs) ... return result является <b>золотым стандартом</b> для написания гибких и надежных декораторов в Python. Запомните эту кострукцию, так как она будет использоваться вами постоянно.

### Задачи

#### Задача 1: Прозрачный декоратор

<b>Условие задачи</b>:

Напишите декоратор transparent_decorator. Этот декоратор не должен добавлять никакого нового поведения. Его единственная задача — <b>корректно</b> принять все аргументы, передать их в декорируемую функцию и вернуть ее результат, не сломав ее работу.

Это задание проверяет ваше понимание базовой универсальной структуры декоратора.

In [13]:
def transparent_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

#### Задача 2: Логгер аргументов

<b>Условие задачи</b>:

Напишите декоратор log_args_and_kwargs. Он должен перед вызовом функции напечатать на экран строку в строгом формате args: (...) | kwargs: {...}, где (...) — это кортеж args, а {...} — словарь kwargs.

In [15]:
def log_args_and_kwargs(func):
    def wrapper(*args, **kwargs):
        print(f'args: {args} | kwargs: {kwargs}')
        return func(*args, **kwargs)
    return wrapper

#### Задача 3: Декоратор, удваивающий результат

<b>Условие задачи</b>:

Напишите универсальный декоратор double_result. Он должен вызвать декорируемую функцию, получить её результат, умножить его на 2 и вернуть. Декоратор должен работать для любой функции, возвращающей число.

In [16]:
def double_result(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

#### Задача 4: Запрет именованных аргументов

<b>Условие задачи</b>:

Напишите декоратор forbid_kwargs. Он должен проверять, были ли в функцию переданы какие-либо именованные аргументы (**kwargs).
- Если словарь kwargs пуст, функция должна выполниться.
- Если в kwargs есть хотя бы один аргумент, декоратор должен выбросить ValueError с сообщением "Keyword arguments are forbidden".

In [18]:
def forbid_kwargs(func):
    def wrapper(*args, **kwargs):
        
        if kwargs:
            raise ValueError('Keyword arguments are forbidden')

        return func(*args)
    return wrapper

#### Задача 5: Полный цикл логирования

<b>Условие задачи</b>:

Напишите декоратор full_cycle_logger. Он должен:
1. Перед вызовом функции напечатать --- START ---.
2. Вызвать функцию и получить ее результат.
3. После вызова напечатать строку в формате Result: {результат}.
4. В самом конце напечатать --- END ---.
5. Вернуть результат, полученный от функции.

In [19]:
def full_cycle_logger(func):
    def wrapper(*args, **kwargs):
        print('--- START ---')
        
        result = func(*args, **kwargs)

        print(f'Result: {result}')
        print('--- END ---')

        return result
    return wrapper

## Сохранение метаданных с functools.wraps

### Шаг 1: Демонстрируем проблему: потеря имени и документации

Мы научились создавать универсальные декораторы, которые работают с любыми аргументами. Они функциональны, но скрывают одну коварную проблему. Наши декораторы, по сути, "воруют" личность у функций, которые они украшают.

Давайте разберемся, что это значит.

#### "Личность" функции: метаданные

В Python любая функция - это объект, и у этого объекта есть специальные атрибуты, которые его описывают. Их называют <b>метаданными</b>. Два самых важных из них:
- \_\_name__: Имя функции (в виде строки).
- \_\_doc__: Строка документации (docstring), которую мы пишем внутри тройных кавычек.

Эти атрибуты крайне важны для отладки, самодокументирования кода и работы автоматических генераторов документации.

#### Как должно быть: смотрим на обычную функцию

Давайте создадим простую функцию и посмотрим на ее "паспортные данные":

In [20]:
def add(a, b):
    """
    Эта функция складывает два числа.
    Она очень важна для нашего проекта!
    """
    return a + b

print(f"Имя функции: {add.__name__}")
print(f"Документация функции: {add.__doc__}")

# Встроенная функция help() также использует эти метаданные
print("\n--- Вывод help(add) ---")
help(add)

Имя функции: add
Документация функции: 
Эта функция складывает два числа.
Она очень важна для нашего проекта!


--- Вывод help(add) ---
Help on function add in module __main__:

add(a, b)
    Эта функция складывает два числа.
    Она очень важна для нашего проекта!



#### Что происходит после применения нашего декоратора?

Теперь давайте возьмем один из наших универсальных декораторов (например, простой логгер) и применим его к нашей функции add.

In [21]:
def simple_logger(func):
    # Добавим docstring обертке для наглядности
    def wrapper(*args, **kwargs):
        """Это документация функции WRAPPER."""
        print(f"Вызываем {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@simple_logger
def add(a, b):
    """
    Эта функция складывает два числа.
    Она очень важна для нашего проекта!
    """
    return a + b

# А теперь снова проверим метаданные!
print(f"Имя функции ПОСЛЕ декоратора: {add.__name__}")
print(f"Документация ПОСЛЕ декоратора: {add.__doc__}")

print("\n--- Вывод help(add) после декоратора ---")
help(add)

Имя функции ПОСЛЕ декоратора: wrapper
Документация ПОСЛЕ декоратора: Это документация функции WRAPPER.

--- Вывод help(add) после декоратора ---
Help on function wrapper in module __main__:

wrapper(*args, **kwargs)
    Это документация функции WRAPPER.



<b>Проблема налицо</b>:
1. <b>Имя потеряно</b>: Наша функция add теперь "думает", что ее зовут wrapper.
2. <b>Документация потеряна</b>: Вместо полезного описания того, что делает функция add, мы видим документацию внутренней функции-обертки.

#### Почему это произошло и почему это плохо?

Это произошло потому, что наш декоратор физически заменил объект функции add на объект функции wrapper. add = simple_logger(add). Теперь имя add указывает на wrapper, и мы видим метаданные wrapper.

<b>Это серьезная проблема</b>:

- <b>Отладка</b>: Если в программе произойдет ошибка, в отчете (traceback) вы увидите непонятное имя wrapper вместо add, что сильно затруднит поиск проблемы. Представьте, у вас 10 декорированных функций, и все они в отчетах об ошибках будут называться wrapper!

- <b>Интерактивная работа и документация</b>: Инструменты вроде help() или автодополнение в Jupyter/IPython перестают быть полезными. Вы больше не сможете быстро посмотреть, что делает функция и какие аргументы она принимает.

#### Итог

Простые декораторы, которые мы писали до сих пор, имеют серьезный побочный эффект: они стирают важные метаданные исходной функции, заменяя их метаданными своей внутренней обертки. Это делает код менее читаемым и сложными для отладки. К счастью, в Python есть встроенное и очень элегантное решение этой проблемы.

### Шаг 2: Объясняем, почему это происходит: подмена личности

В предыдущем шаге мы увидели неприятный эффект: после декорирования наша функция add потеряла свое имя и документацию. Чтобы понять, почему это происходит, нам нужно вспомнить самую суть механизма работы декоратора.

#### Вспомним, что скрывается за символом @

Когда мы пишем этот код:

In [23]:
# @my_decorator
# def my_function():
#     pass

Мы просто используем "синтаксический сахар" для вот такой, более явной операции:

In [24]:
# def my_function():
#     pass

# my_function = my_decorator(my_function)

Эта строка my_function = my_decorator(my_function) - ключ к разгадке.

#### Давайте проследим за объектами шаг за шагом

1. <b>Создание исходной функции</b>: Python выполняет блок def add(...). В этот момент в памяти создается объект-функция. У этого объекта есть атрибуты, например:
    - \_\_name__ = 'add'
    - \_\_doc__ = 'Эта функция складывает два числа...'

2. <b>Вызов декоратора</b>: Сразу после этого Python выполняет операцию декорирования: simple_logger(add). Мы передаем наш объект-функцию add внутрь декоратора simple_logger.

3. <b>Создание обертки</b>: Внутри simple_logger создается совершенно новый объект-функция - наша wrapper. У этого нового объекта, разумеется, есть свои собственные метаданные:
    - \_\_name__ = 'wrapper'
    - \_\_doc__ = 'Это документация функции WRAPPER.'

4. <b>Возврат и перезапись</b>: Декоратор simple_logger завершает свою работу и возврщает объект-функцию wrapper. И вот самый важный момент. Python выполняет присваивание: add = (то, что вернул декоратор). Теперь имя add в нашей программе <b>перестает ссылаться на исходную функцию и начинает ссылаться на новый объект-функцию wrapper</b>.

#### Визуализация подмены

Можно представить это так:

##### ДО ДЕКОРИРОВАНИЯ

In [25]:
# Имя         Объект в памяти
# +-------+     +-------------------------------+
# |  add  | --> | function object 'add'         |
# +-------+     |   __name__: 'add'             |
#               |   __doc__: 'Эта функция...'   |
#               +-------------------------------+

##### ПОСЛЕ ДЕКОРИРОВАНИЯ

In [27]:
# Имя         Объект в памяти
# +-------+     +-------------------------------+
# |  add  | --+ | function object 'add'         |  <-- Исходный объект не удаляется!
# +-------+     |   __name__: 'add'             |      Он продолжает "жить" внутри
#               |   __doc__: 'Эта функция...'   |      обёртки wrapper благодаря замыканию.
#               +-------------------------------+      Но прямая ссылка по имени 'add' потеряна.
#               |
#               +-> +---------------------------------+
#                   | function object 'wrapper'       |
#                   |   __name__: 'wrapper'           |
#                   |   __doc__: 'Это документация...'|
#                   +---------------------------------+

Старый объект add не удаляется сборщиком мусора, потому что на него остается ссылка изнутри функции wrapper (это называется <b>замыканием</b>). Однако <b>прямой доступ</b> к исходному объекту по имени add мы потеряли. Теперь это имя указывает на wrapper.

#### Итог

Потрея метаданных - это не какой-то странный баг, а прямое и логичное следствие самого механизма работы декортара. <b>Декортаор заменяет исходную фукнцию всоей внутренней функцией-оберткой</b>. Поэтому, когда мы обращаемся к add.\_\_name__, мы, по факту, обращаемся к wrapper.\_\_name__.

Понимание этой "подмуны" очень важно. Теперь мы готовы к тому, чтобы научиться делать эту подмену "умной" - так, чтобы обертка не просто заменяла исходную функцию, а маскировалась под нее, копируя ее "паспортные данные".

### Шаг 3: Представляем решение - декоратор @functools.wraps

Проблема с потерей метаданных настолько распространена и важна, что в стандартной библиотеке Python для нее есть готовое решение. Это специальный декоратор, предназначенный для использования <b>внутри</b> других декораторов.

Встречайте - <b>@functools.wraps</b>

#### Что это такое и как он работает?

@wraps - это сам по себе декоратор. Его задача - помочь нам "обмануть" систему, скопировав все важные метаданные (\_\_name__, \_\_doc__, \_\_module__ и другие) из исходной, оборачиваемой функции в нашу функцию-обертку.

Проще говоря, @wraps делает так, чтобы наша wrapper "притворилась" исходной функцией func.

#### Как его использовать?

Использование @wraps предельно простое.
1. Сначаала его нужно импортировать из модуля functools.
2. Затем его нужно применить <b>как декоратор к нашей внутренней функции wrapper</b>.
3. В @wraps нужно передать один аргумент - ссылку на исходную функцию, которую мы декорируем (func).

#### Модернизируем наш декоратор

Давайте возьмем наш "проблемный" декоратор simple_logger и исправим его с помощью @wraps.

##### "ДО": Проблемная версия

In [28]:
def simple_logger(func):
    def wrapper(*args, **kwargs):
        """Это документация функции WRAPPER."""
        print(f"Вызываем {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

##### "ПОСЛЕ": Исправленная, профессиональная версия

In [29]:
from functools import wraps # 1. Импортируем wraps

def professional_logger(func):
    # 2. Применяем @wraps к нашей обертке
    # 3. Передаем в него исходную функцию 'func'
    @wraps(func) 
    def wrapper(*args, **kwargs):
        """
        Эта документация теперь будет игнорироваться,
        так как @wraps заменит ее на документацию из 'func'.
        """
        print(f"Вызываем {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

Всего две небольшие правки, но они кардинально меняют дело!

#### Демонстрация исправленного поведения

Теперь давайте применим наш новый, "профессиональный" декоратор к функции add и снова проверим ее метаданные.

In [30]:
@professional_logger
def add(a, b):
    """
    Эта функция складывает два числа.
    Она очень важна для нашего проекта!
    """
    return a + b

# Проверяем метаданные еще раз
print(f"Имя функции ПОСЛЕ 'правильного' декоратора: {add.__name__}")
print(f"Документация ПОСЛЕ 'правильного' декоратора: {add.__doc__}")

print("\n--- Вывод help(add) после 'правильного' декоратора ---")
help(add)

Имя функции ПОСЛЕ 'правильного' декоратора: add
Документация ПОСЛЕ 'правильного' декоратора: 
Эта функция складывает два числа.
Она очень важна для нашего проекта!


--- Вывод help(add) после 'правильного' декоратора ---
Help on function add in module __main__:

add(a, b)
    Эта функция складывает два числа.
    Она очень важна для нашего проекта!



<b>Победа!</b> Наша декорированная функция теперь имеет точно такие же имя и документацию, как и исходная. Отладчики, генераторы документации и вызов help() снова работают правильно.

#### Итог

Использования @functools.wraps - это <b>золотой стандарт</b> и обязательная практика при написании любых декораторов в Python. Он решает проблему потери метаданных просто, элегантно и надежно.

<b>Запомните правило</b>: всегда, когда вы пишете декоратор, который заменяет одну функцию другой, применяйте @wraps(func) к вашей внутренней функции-обертке. Это признак качественного и профессионального кода.

### Задачи

#### Задача 1: Исправить потерю имени

<b>Условие задачи</b>:

Вам дан "сломанный" декоратор broken_decorator, который теряет имя декорируемой функции.
Импортируйте wraps из модуля functools и исправьте декоратор так, чтобы он сохранял метаданные исходной функции.

In [31]:
from functools import wraps

def broken_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        """Это документация обертки."""
        print("Wrapper is called")
        return func(*args, **kwargs)
    return wrapper

#### Задача 2: Исправить потерю документации

<b>Условие задачи</b>:

Используя тот же подход, что и в прошлой задаче, исправьте декоратор decorator_with_wrong_doc, чтобы он сохранял строку документации (__doc__) исходной функции.

In [39]:
from functools import wraps

def decorator_with_wrong_doc(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        """Неправильная документация."""
        return func(*args, **kwargs)
    return wrapper

@decorator_with_wrong_doc
def function_with_docstring():
    """Это правильная строка документации."""
    pass

#### Задача 3: Написать "правильнй" декоратор с нуля

<b>Условие задачи</b>:

Напишите "правильный" декоратор perfect_decorator, который сохраняет метаданные.
Декоратор должен:
1. Перед вызовом функции печатать --- Before ---.
2. После вызова функции печатать --- After ---.
3. Корректно сохранять \_\_name__ и \_\_doc__ декорируемой функции.

In [ ]:
from functools import wraps

def perfect_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('--- Before ---')
        result = func(*args, **kwargs)
        print('--- After ---')
        return result
    return wrapper

#### Задача 4: Проверка работы help()

<b>Условие задачи</b>:

В ::header скрыт код, который переопределяет встроенную функцию help() так, что она просто печатает \_\_name__ и \_\_doc__ объекта.
Напишите декоратор helpful_decorator, который сохраняет метаданные, чтобы вызов help() для декорированной функции работал корректно. Логика самого декоратора не важна, он может просто вызывать функцию.

In [ ]:
from functools import wraps

def helpful_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result
    return wrapper

#### Задача 5: Стек из "правильных" декораторов

<b>Условие задачи</b>:

Напишите два "правильных" декоратора (сохраняющих метаданные): decorator_a и decorator_b.
- decorator_a должен печатать A перед вызовом.
- decorator_b должен печатать B перед вызовом.

Примените их к функции target_func так, чтобы @decorator_a был внешним, а @decorator_b — внутренним.

Задача проверяет, что \_\_name__ сохраняется даже при множественном декорировании.



In [41]:
from functools import wraps

def decorator_a(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('A')
        result = func(*args, **kwargs)
        return result
    return wrapper

def decorator_b(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('B')
        result = func(*args, **kwargs)
        return result
    return wrapper

## Декораторы с аргументами

### Шаг 1: Объясняем, что иногда хочется настраивать поведение самого декоратора

До сих пор все наши декораторы были похожи на выключатель: @timer либо включен, либо выключен. Он всегда делает одно и то же - выводит время в секундах в консоль. @logger всегда выводит лог в одном и том же формате.

Они полезны, но их поведение "зашито" прямо в код. А что, если нам нужна большая гибкость?

#### Давайте представим несколько сценариев "хотелось бы..."?

- <b>Для нашего @timer'а</b>: Хотелось мы иметь возможность выбирать единицы измерения. Например, один раз измерить в секундах, а для другой, очень быстрой функции, - в миллисекундах.

    - timer(units='seconds')

    - timer(units='milliseconds')

- <b>Для нашего @logger'а</b>: Хотелось мы указывать, в какой файл записывать лог, вместо вывода в консоль.

    - @logger(logfile='database_operations.log')

- <b>Для кэширования</b>: Мы уже видели это в стандартной библиотеке! @lru_cache не просто включал кэширование, он позволял его <b>настроить</b>, указав максимальный размер кэша: 

    - @lru_cache(maxsize=128)

#### Самый наглядный пример: декоратор для повторных попыток (@retry)

Представьте, вы пишете функцию, которая обращается к нестабильному сетевому ресурсу (например, скачивает данные с сайта или подключается к базе данных). Такой запрос может иногда не сработать с первого раза.

Было бы здорово иметь декоратор @retry, который в случае ошибки не падает, а автоматически пытается выполнить функцию еще раз. Но тут же возникает вопрос:

"А <b>сколько</b> раз пытаться? И нужно ли делать паузу между попытками?"

Очевидно, что "жестко" зашить эти параметры в код декоратора - плохая идея. Для одной функции нам может быть достаточно 2 попытки, а для другой, более критичной, - 5 попыток с задержкой в 2 секунды.

#### Формулируем новую цель

Мы хотим научиться создавать <b>параметризованные декораторы</b>, то есть декораторы, в которые можно передавать аргументы для настройки их собственного поведения. Наша цель - научиться писать код, который позволит нам делать вот так:

In [2]:
# # Это то, чего мы хотим достичь:
# @retry(times=3, delay=1) # Попробовать 3 раза, с паузой в 1 секунду
# def connect_to_database():
#     print("Пытаюсь подключиться к БД...")
#     # ... здесь код, который может вызвать исключение ...
#     raise ConnectionError("Не удалось подключиться")

# connect_to_database()

Обратите внимание на синтаксис: @retry(times=3, delay=1). <b>Ключевое отличие - это круглые скобки () после имени декоратора</b>.

- @retry - это применение обычного декоратора.

- @retry(...) - это <b>вызов</b> чего-то, что <b>вернет</b> нам декоратор.

#### Итог

Чтобы сделать наши декораторы по-настоящему мощными и переиспользуемыми, нам нужен мезанизм для их настройки. Мы хотим передавать параметры не только в декорируемую функцию, но и в сам декоратор. В следующих шагах мы разберем, какая трехуровневая структура ("фабрика декораторов") позволяет нам этого добиться.

### Шаг 2: Показываем структуру "фабрики декораторов"

Как мы заметили в предыдущем шаге, когда мы пишем @retry(times=3), мы фактически <b>вызываем</b> функцию retry. А что должен вернуть наш вызов? Он должен вернуть тот самый декоратор, который затем будет применен к нашей функции (connect_to_database).

Это приводит нас к новой, трехуровневой структуре:

1. <b>Фабрика декораторов</b>: Самая внешняя функция. Она принимает аргументы для настройки (times=3, delay=1). Ее единственная задача - создать и <b>вернуть</b> настоящий декоратор.

2. <b>Декоратор</b>: Средний уровень. Это то, что мы писали раньше. Он принимает в качестве аргумента декорируемую <b>функцию</b> (func) и возвращает функцию-обертку.

3. <b>Обертка</b>: Самый внутренний уровень. Эта функция выполняет всю работу: она принимает аргументы вызова (*args, **kwargs), реализует новую логику (например, цикл for для повторных попыток) и вызывает исходную функцию.

#### Давайте построим "скелет" этой структуры

Мы еще не будем реализовывать логику повторов, а просто посмотрим на вложенность функций и на то, что каждый из них принимает и возвращает.

In [3]:
# Уровень 1: ФАБРИКА
def retry(times, delay):
    """
    Принимает аргументы для настройки.
    Возвращает готовый декоратор.
    """
    print(f"--- Фабрика retry вызвана с параметрами: times={times}, delay={delay} ---")
    
    # Уровень 2: ДЕКОРАТОР
    def decorator(func):
        """
        Принимает декорируемую функцию.
        Возвращает обертку.
        """
        print(f"--- Декоратор применен к функции: {func.__name__} ---")
        
        # Уровень 3: ОБЕРТКА
        def wrapper(*args, **kwargs):
            """
            Выполняет всю работу.
            """
            print(f"--- Обертка wrapper вызвана! (Здесь будет логика повторов) ---")
            
            # Внутри wrapper мы имеем доступ и к 'times', и к 'delay',
            # и к 'func', и к '*args', '**kwargs' благодаря замыканиям!
            print(f"    Параметры из фабрики: times={times}, delay={delay}")
            
            # Вызываем исходную функцию
            return func(*args, **kwargs)
            
        return wrapper # Декоратор возвращает обертку
        
    return decorator # Фабрика возвращает декоратор

#### Проследим за процессом шаг за шагом

Давайте посмотрим, что происходит, когда Python встречает этот код:

In [4]:
@retry(times=3, delay=1)
def unstable_function():
    print("Выполняю важную, но нестабильную операцию...")

--- Фабрика retry вызвана с параметрами: times=3, delay=1 ---
--- Декоратор применен к функции: unstable_function ---


1. <b>Вызов Фабрики</b>: Python видит retry(...) и <b>сразу же вызывает</b> эту функцию: retry(tims=3, delay=1).
    
    - В консоли напечатается: --- Фабрика retry вызвана... ---

    - retry создает и возвращает внутреннюю функцию decorator. <b>На этом этапе retry завершает свою работу!</b>

2. <b>Применение Декоратора</b>: Теперь Python ведет себя так, как будто мы написали @decorator. Он берет нашу функцию unstable_function и передает ее в только что созданный decorator.

    - В консоли напечатается: --- Декоратор применен к функции: unstable_function ---

    - decorator создает и возвращает внутреннюю функцию wrapper. wrapper "помнит" и func (то есть unstable_function), и times, и delay благодаря замыканиям.

3. <b>Финальная замена</b>: Имя unstable_function теперь ссылается на wrapper.

Весь этот процесс происходит <b>один раз</b>, когда Python загружает наш скрипт.

#### А что происходит при вызове?

In [5]:
unstable_function()

--- Обертка wrapper вызвана! (Здесь будет логика повторов) ---
    Параметры из фабрики: times=3, delay=1
Выполняю важную, но нестабильную операцию...


Теперь, когда мы вызываем unstable_function(), мы на самом деле вызываем wrapper().

#### Итог

Чтобы создать декоратор с аргументами, нам нужна "фабрика" - дополнительный, самый внешний уровень вложенности. Эта функция-фабрика принимает параметры для настройки и возвращает обычный декоратор, который мы уже умеем писать. Вся магия этой конструкции работает благодаря <b>замыканиям</b>, так как самая внутренняя функция wrapper получает доступ к переменным из всех окружающих ее областей видимости.

### Шаг 3: Разбираем трехуровневую вложенность: Фабрика -> Декоратор -> Обертка

Чтобы понять эту структуру, давайте дадим каждой из трех функций четкую роль и разберемся, когда она выполняется и за что отвечает.

Представим себе нашу конструкцию:

In [7]:
def retry(times, delay):  # <-- УРОВЕНЬ 1: ФАБРИКА
    
    def decorator(func):      # <-- УРОВЕНЬ 2: ДЕКОРАТОР
        
        @wraps(func)
        def wrapper(*args, **kwargs): # <-- УРОВЕНЬ 3: ОБЕРТКА
            # ... основная логика ...
            ...
        
        return wrapper
        
    return decorator

#### Уровень 1: Фабрика (retry)

- <b>Ее работа</b>: Конфигурация. Это "менеджер", который принимает заказ. Его единственная задача - получить параметры (times, delay) и на их основе создать специализированный декоратор.

- <b>Что принимает</b>? Аргументы для <b>настройки самого декоратора</b>.

- <b>Что возвращает</b>? Готовую функцию-декоратор (decorator).

- <b>Когда выполняется? ОДИН РАЗ</b>, в момент, когда Python читает строку @retry(times=3, delay=1). Это проиходит при загрузке модуля, а не при вызове декорируемой функции.

<b>Аналогия</b>: Вы приходите на завод и говорите: "Мне нужен станок, который будет штамповать детали по 3 раза". retry(times=3) - это ваш заказ. Завод (фабрика) не штампует детали сам, <b>он собирает и возвращает вам станок (decorator)</b>, настроенный именно на 3 попытки.

#### Уровень 2: Декоратор (decorator)

- <b>Его работа</b>: Связывание. Это "сборочный цех" или тот самый "станок", который мы получили от фабрики. Его задача - взять "заготовку" (нашу исходную функцию func) и соединить ее с логикой, которую мы будем выполнять.

- <b>Что принимает? Декорируемую функцию</b> (func).

- <b>Что возвращает?</b> Финальную, готовую к работе функцию-обертку (wrapper).

- <b>Когда выполняется? ОДИН РАЗ</b>, сразу после фабрики. Это результат применения символа @ к тому, что вернула фабрика.

<b>Аналогия</b>: Вы берете свой станок (decorator) и вставляете в него пресс-форму для нужной детали (передаете ему func). Станок теперь готов к работе. Результатом является готовый к запуску производстенный цикл (wrapper).

#### Уровень 3: Обертка (wrapper)

- <b>Ее работа</b>: Исполнение. Это "рабочий", который непосредственно выполняет всю логику. Он знает, сколько раз нужно повторить операцию (из times), какую функцию нужно вызывать (func) и с какими аргументами.

- <b>Что принимает?</b> Аргументы для <b>вызова декорируемой функции</b> (*args, **kwargs).

- <b>Что возвращает?</b> Результат выполнения декорируемой функции.

- <b>Когда выполняется? КАЖДЫЙ РАЗ</b>, когда мы вызываем нашу декорированную функцию в коде (например, untable_function()).

<b>Аналогия</b>: Вы нажимаете зеленую кнопку "Старт" на станке. wrapper запскается, берет сырье (*args, **kwargs), пытается проштамповать деталь (func(*args, **kwargs)), в случае неудачи пробует еще раз (используя times), и в конце отдает вам готовую деталь (возвращает result).

#### Как они видят друг друга? Магия замыканий

- <b>Обертка (wrapper)</b> - самая "богатая". Благодаря замыканиям, она видит всё:
    
    - times и delay из области видимости Фабрики.

    - func из области видимости Декоратора.

    - *args и **kwargs из своих собственных аргументов. Именно поэтому она может выполнить всю необходимую работу.

#### Итог

Запомните эту трехуровневую структуру и ее назначение, и вы сможете писать любые параметризованные декораторы.

- <b>Фабрика (@имя(...))</b> - для <b>настройки</b>. Выполняется один раз при определении.

- <b>Декоратор (@...)</b> - для <b>связывания</b> с функцией. Выполняется один раз при определении.

- <b>Обертка (имя_функции())</b> - для <b>выполнения</b> работы. Выполняется каждый раз при вызове.

### Задачи

#### Задача 1: Декоратор-повторитель

<b>Условие задачи</b>:

Напишите фабрику декораторов repeat(times).

Она должна принимать один аргумент times (целое число) и возвращать декоратор. Этот декоратор, в свою очередь, должен заставлять декорируемую функцию выполниться times раз.

In [8]:
from functools import wraps

def repeat(times: int):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

#### Задача 2: Декоратор-префикс

<b>Условие задачи</b>:

Напишите фабрику декораторов prefix(text).

Она должна принимать один строковый аргумент text. Возвращаемый декоратор должен перед вызовом функции печатать этот text на экран.

In [10]:
from functools import wraps

def prefix(text: str):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(text)
            result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

#### Задача 3: Декоратор, умножающий результат

<b>Условие задачи</b>:

Напишите фабрику декораторов multiply_by(factor).

Она должна принимать один числовой аргумент factor. Возвращаемый декоратор должен вызвать декорируемую функцию, получить ее результат, умножить его на factor и вернуть.

In [12]:
from functools import wraps

def multiply_by(factor):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs) * factor
        return wrapper
    return decorator

#### Задача 4: Валидатор типа для конкретного аргумента

<b>Условие задачи</b>:

Напишите фабрику декораторов validate_type(arg_name, expected_type).
Она должна принимать два аргумента:
- arg_name: имя аргумента функции, который нужно проверить (строка).
- expected_type: ожидаемый тип этого аргумента.

Возвращаемый декоратор должен проверять, что именованный аргумент с именем arg_name имеет тип expected_type. Если тип не совпадает, нужно выбросить TypeError.

Подсказка: искомый аргумент будет в словаре kwargs внутри обертки.

In [13]:
from functools import wraps

def validate_type(arg_name, expected_type):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if not isinstance(kwargs[arg_name], expected_type):
                raise TypeError()
            return func(*args, **kwargs)
        return wrapper
    return decorator

#### Задача 5: Декоратор с задержкой

<b>Условие задачи</b>:

Напишите фабрику декораторов delay(seconds).

Она должна принимать один аргумент seconds (число). Возвращаемый декоратор должен делать паузу на seconds секунд <b>перед</b> вызовом декорируемой функции.

Подсказка: используйте time.sleep().

In [14]:
from functools import wraps
from time import sleep

def delay(seconds):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            sleep(seconds)
            return func(*args, **kwargs)
        return wrapper
    return decorator

## Вложенные (стекируемые) декораторы

### Шаг 1: Показываем, как можно применить несколько декораторов к одной функции

Мы научились создавать разнообразные декораторы, каждый из которых решает одну конкретную задачу: один измеряет время, другой логирует вызовы, третий кэширует результаты.

Но что, если для одной функции нам нужно <b>сразу несколько</b> таких "улучшений"? Например, мы хотим, чтобы вызов функции одновременно и логировался, и измерялся по времени.

К счастью, Python предоставляет для этого очень простой и интуитивно понятный синтаксис. Мы можем просто "сложить" или <b>"стекировать"</b> (stack) декораторы дргу на друга, располагая их прямо над определением функции.

#### Как это выглядит синтаксически?

Вы просто пишете несколько декораторов, каждый на своей строке:

In [15]:
# @decorator_A
# @decorator_B
# @decorator_C
# def my_function():
#     # ... тело функции ...
#     pass

#### Давайте рассмотрим это на практическом примере

Предположим, у нас есть два готовых декоратора из предыдущих уроков: @timer и @logger.

In [16]:
from functools import wraps
import time

# --- Наши готовые декораторы ---

def logger(func):
    """Логирует вызов функции."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG]: Вызывается функция {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"[LOG]: Функция {func.__name__} завершила работу.")
        return result
    return wrapper

def timer(func):
    """Измеряет время выполнения функции."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start_time
        print(f"[TIMER]: Функция {func.__name__} выполнялась {duration:.4f} секунд.")
        return result
    return wrapper

# --- Применяем ОБА декоратора к одной функции ---

@logger
@timer
def process_data(delay_seconds):
    """
    Имитирует сложную обработку данных с задержкой.
    """
    print(f"    (Начинаю обработку, жду {delay_seconds} сек...)")
    time.sleep(delay_seconds)
    print("    (...обработка завершена)")
    return "Данные успешно обработаны"

# --- Вызываем нашу "дважды украшенную" функцию ---
process_data(1.5)

[LOG]: Вызывается функция process_data...
    (Начинаю обработку, жду 1.5 сек...)
    (...обработка завершена)
[TIMER]: Функция process_data выполнялась 1.5008 секунд.
[LOG]: Функция process_data завершила работу.


'Данные успешно обработаны'

#### Почему это так мощно? (Принцип композиции)

Эта возможность позволяет нам следовать одному из лучших принципов программирования - <b>композиции</b>. Вместо того чтобы писать один огромный, сложный декоратор, который делает всё сразу, мы пишем несколько маленьких, простых и сфокусированных декораторов, каждый из которых делает что-то одно, но делает это хорошо.

Затем мы можем комбинировать их, как кубики Lego, создавая именно то поведение, которое нам нужно для каждой конкретной функции.

#### Что на самом деле происходит "под капотом"?

Когда Python видит стек из нескольких декораторов, он применяет их последовательно, <b>снизу вверх</b> (или изнутри наружу).

In [17]:
# @logger
# @timer
# def process_data(...):
#     ...

является синтаксическим сахаром для следующего выражения:

In [18]:
# process_data = logger(timer(process_data))

Сначала timer оборачивает process_data, а затем logger оборачивает то, что получилось в результате (timer(process_data)).

#### Итог

Мы можем применять к одной функции неограниченное количество декораторов, просто располагая их друг над другом. Это мощный прием, который позволяет комбинировать функциональность и писать чистый, модульный код. Порядок, в котором мы располагаем декораторы, имеет значение, так как они применяются последовательно.

### Шаг 2: Объясгяем и иллюстрируем порядок применения и выполнения декораторов

Когда мы "стекируем" декораторы, очень важно понимать, в каком порядке они применяются к функции и в каком порядке выполняется их код. Здесь есть два связанных, но разных процесса: <b>применение</b> и <b>выполнение</b>.

#### Правило №1: Применение происходит СНИЗУ ВВЕРХ

Как м выяснили в прошлом шаге, запись:

In [21]:
# @A
# @B
# def my_func():
#     pass

...эквивалентна следующей:

In [22]:
# my_func = A(B(my_func))

Если посмотреть на это выражение, становится очевидно:

1. Сначала Python выполняет то, что в самых внутренних скобках: B(my_func). То есть, декоратор @B, который находится <b>ближе всего к функции</b>, применяется первым.

2. Затем декоратор @A применяется к <b>результату</b> предыдущей операции: A(...).

Таким образом, декораторы "надеваются" на функцию как слои одежды: сначала тот, что ближе к телу (@B), потом тот, что дальше (@A).

#### Правило №2: Выполнение присходит СВЕРЗУ ВНИЗ (как "матрешка")

Хотя применяются декораторы снизу вверх, их код во время вызова функции выполняется <b>сверху вниз</b>. Внешний декоратор начинает работать первым, затем он передает управление внутреннему, и так далее, пока не будет вызвана сама исходная функция. После того как функция отработала, управление возвращается в обратном порядке.

Это проще всего понять на примере.

#### Иллюстрация порядка выполнения

Давайте создадим два очень простых декоратора, которые будут просто печатать, когда их код "до" и "после" выполняется.

In [1]:
from functools import wraps

def decorator_one(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("[ДЕКОРАТОР ONE]: Перед вызовом")
        result = func(*args, **kwargs)
        print("[ДЕКОРАТОР ONE]: После вызова")
        return result
    return wrapper

def decorator_two(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("[ДЕКОРАТОР TWO]: Перед вызовом")
        result = func(*args, **kwargs)
        print("[ДЕКОРАТОР TWO]: После вызова")
        return result
    return wrapper

##### Сценарий 1: @decorator_one сверху, @decorator_two снизу

In [2]:
print("--- Сценарий 1: @one сверху, @two снизу ---")
@decorator_one
@decorator_two
def say_hello():
    print("    >> Выполняется тело функции say_hello...")

say_hello()

--- Сценарий 1: @one сверху, @two снизу ---
[ДЕКОРАТОР ONE]: Перед вызовом
[ДЕКОРАТОР TWO]: Перед вызовом
    >> Выполняется тело функции say_hello...
[ДЕКОРАТОР TWO]: После вызова
[ДЕКОРАТОР ONE]: После вызова


<b>Анализ</b>: Код выполняется как "матрешка" или "лук".

1. Запускается самый внешний слой (ONE). Он печатает "Перед вызовом".

2. ONE передает управление следующему слою (TWO). TWO печатает "Перед вызовом".

3. TWO передает управление самой функции say_hello. Она выполняется.

4. say_hello завершается, управление возвращается к TWO. Он печатает "После вызова".

5. TWO завершается, управление возвращается к ONE. Он печатает "После вызова".

#### Сценарий 2: Меняем декораторы местами

In [3]:
print("\n--- Сценарий 2: @two сверху, @one снизу ---")
@decorator_two
@decorator_one
def say_goodbye():
    print("    >> Выполняется тело функции say_goodbye...")

say_goodbye()


--- Сценарий 2: @two сверху, @one снизу ---
[ДЕКОРАТОР TWO]: Перед вызовом
[ДЕКОРАТОР ONE]: Перед вызовом
    >> Выполняется тело функции say_goodbye...
[ДЕКОРАТОР ONE]: После вызова
[ДЕКОРАТОР TWO]: После вызова


<b>Анализ</b>: Порядок изменился! Теперь самый внешний слой - это @decorator_two, и именно он запускается первым и завершается последним.

#### Аналогия: Упаковка подарка

- <b>Подарок</b> - это ваша исходная функция (say_hello).

- <b>Первый слой оберточной бумаги</b> - это нижний декоратор (@decorator_two). Вы заворачиваете подарок в него первым.

- <b>Ленточка</b> - это верхний декоратор (@decorator_one). Вы повязываете ее поверх бумаги.

- <b>Выполнение функции</b> - это распаковка подарка. Вы <b>сначала снимаете ленточку (ONE)</b>, затем <b>разворачиваете бумагу (TWO)</b>, и только потом добираетесь до <b>подарка (say_hello)</b>.

#### Итог

Порядок декораторов в стеке имеет решающее значение.

- <b>Применение (однократно, при загрузке)</b>: Снизу вверх.

- <b>Выполнение (каждый раз при вызове)</b>: Сверху вниз, как слои луковицы.

Понимая это правило, вы можете соознанно комбинировать декораторы для достижения нужного вам поведения. Например, если бы @logger был снаружи, а @timer внутри, то @logger залогировал бы общее время работы, включая работу @timer. Если наоборот - @timer измерил бы время работы, включая работу @logger.

### Шаг 3: Рассматриваем наглядный пример и анализируем порядок вывода

Чтобы увидеть разницу в поведении, мы создадим одну и ту же функцию, но применим к ней декораторы в разном порядке. Наша функция будет имитировать загрузку отчета, то есть просто делать паузу на 1 секунду.

#### Подготовим наши инструменты:

In [4]:
from functools import wraps
import time

def logger(func):
    """Логирует начало и конец вызова функции."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOGGER]: Начало вызова '{func.__name__}'")
        result = func(*args, **kwargs)
        print(f"[LOGGER]: Конец вызова '{func.__name__}'")
        return result
    return wrapper

def timer(func):
    """Измеряет время выполнения функции."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start_time
        print(f"[TIMER]:  '{func.__name__}' выполнилась за {duration:.2f} сек.")
        return result
    return wrapper

#### Сценарий 1: @logger (внешний), @timer (внутренний)

В этом случае @logger - это самый верхний, внешний слой, а @timer - внутренний, ближе к функции.

In [5]:
print("--- Сценарий 1: @logger сверху ---")
@logger
@timer
def fetch_report():
    """Имитирует загрузку отчета."""
    print("    ...загрузка отчета...")
    time.sleep(1)
    print("    ...отчет готов!")

fetch_report()

--- Сценарий 1: @logger сверху ---
[LOGGER]: Начало вызова 'fetch_report'
    ...загрузка отчета...
    ...отчет готов!
[TIMER]:  'fetch_report' выполнилась за 1.00 сек.
[LOGGER]: Конец вызова 'fetch_report'


<b>Анализ</b>:

1. <b>Начинается вызов</b>: Управление получает самый внешний декоратор - @logger. Он печатает [LOGGER]: Начало вызова...

2. <b>Передача управления</b>: @logger вызывает свою "обернутую" функцию, которая на самом деле является результатом работы @timer.

3. <b>Запускается таймер</b>: Управление получает @timer. Он <b>засекает время начала</b>, но пока ничего не печатает.

4. <b>Выполнение функции</b>: @timer вызывает свою "обернутую" функцию - нашу исходную fetch_report(). Она выполняется, печатая сообщение о загрузке и делая паузу.

5. <b>Остановка таймера</b>: fetch_report() завершается. Управление возвращается к @timer. Он <b>останавливает таймер</b>, вычисляет разницу и печатает [TIMER]: ... выполнилась за ...

6. <b>Завершение</b>: @timer завершается, и управление возвращается к самому внешнему декоратору @logger. Он печатает [LOGGER]: Конец вызова ...

<b>Вывод по Сценарию 1</b>: @logger логирует <b>весь процесс целиком</b>, включая работу @timer.

#### Сценарий 2: @timer (внешний), @logger (внутренний)

Теперь поменяем их местами. @timer станет внешним слоем.

In [6]:
print("\n--- Сценарий 2: @timer сверху ---")
@timer
@logger
def download_file():
    """Имитирует загрузку файла."""
    print("    ...загрузка файла...")
    time.sleep(1)
    print("    ...файл загружен!")

download_file()


--- Сценарий 2: @timer сверху ---
[LOGGER]: Начало вызова 'download_file'
    ...загрузка файла...
    ...файл загружен!
[LOGGER]: Конец вызова 'download_file'
[TIMER]:  'download_file' выполнилась за 1.00 сек.


<b>Анализ</b>:

1. <b>Начинается вызов</b>: Управление получает самый внешний декоратор — @timer. Он <b>засекает время начала</b>.

2. <b>Передача управления</b>: @timer вызывает свою "обернутую" функцию — результат работы @logger.

3. <b>Запускается логгер</b>: Управление получает @logger. Он печатает [LOGGER]: Начало вызова....

4. <b>Выполнение функции</b>: @logger вызывает download_file(). Она выполняется.

5. <b>Завершение логгера</b>: download_file() завершается. Управление возвращается к @logger. Он печатает [LOGGER]: Конец вызова....

6. <b>Остановка таймера</b>: @logger завершается. Управление возвращается к самому внешнему @timer. Он <b>останавливает таймер</b> и печатает результат.

<b>Вывод по Сценарию 2</b>: @timer измеряет время выполнения <b>всего, что внутри него</b>, то есть и работу @logger, и работу самой функции. Сообщения логгера оказываются "внутри" временного промежутка, который измеряет таймер.

#### Итог

Порядок декораторов напрямую влияет на то, что "видит" каждый из них. Внешний декоратор "оборачивает" и контролирует все внутренние. В зависимости от того, что для вас важнее - замерить "чистое" время работы функции или залогировать всю операцию целиком - вы будете выбирать тот или иной порядок их расположения.

### Задачи

#### Задача 1: Написать два декоратора

<b>Пояснение</b>:

Нужно написать два стандартных декоратора и применить их к функции. Чтобы @decorator_a был внешним, его нужно написать выше (дальше от def). Порядок выполнения кода "до" будет соответствовать порядку написания декораторов: сначала A, потом B.

In [7]:
from functools import wraps

def decorator_a(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('A')
        return func(*args, **kwargs)
    return wrapper

def decorator_b(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('B')
        return func(*args, **kwargs)
    return wrapper

#### Задача 2: Порядок выполнения "до"

<b>Условие задачи</b>:

Вам даны два готовых декоратора: print_first и print_second.
Примените их к функции main_action в таком порядке, чтобы при ее вызове сначала напечаталось "First", а затем "Second".

Подсказка: код "до" выполняется в том порядке, в котором декораторы написаны сверху вниз.

In [8]:
from functools import wraps

def print_first(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("First")
        return func(*args, **kwargs)
    return wrapper

def print_second(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Second")
        return func(*args, **kwargs)
    return wrapper

#### Задача 3: Порядок выполнения "после"

<b>Условие задачи</b>:

Вам даны два готовых декоратора: print_after_first и print_after_second. Каждый из них печатает свое сообщение после вызова функции.

Примените их к функции core_logic в таком порядке, чтобы при ее вызове сначала напечаталось "Logic", затем "Second After", и только в самом конце "First After".

In [9]:
from functools import wraps

def print_after_first(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print("First After")
        return result
    return wrapper

def print_after_second(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print("Second After")
        return result
    return wrapper

#### Задача 4: Декоратор, изменяющий результат

<b>Условие задачи</b>:

Вам даны два декоратора: @add_one (прибавляет 1 к результату) и @multiply_by_two (умножает результат на 2).

Примените их к функции get_number в таком порядке, чтобы итоговый результат вызова get_number() был равен <b>12</b>.

Исходная функция возвращает 5.

In [10]:
from functools import wraps

def add_one(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + 1
    return wrapper

def multiply_by_two(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

#### Задача 5: Полный цикл "Матрешка"

<b>Условие задачи</b>:

Напишите два декоратора: @outer и @inner.
- @outer должен печатать Outer Before до и Outer After после вызова.
- @inner должен печатать Inner Before до и Inner After после вызова.

Примените их к функции action так, чтобы @outer был внешним слоем.

In [11]:
from functools import wraps

def outer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('Outer Before')
        result = func(*args, **kwargs)
        print('Outer After')
        return result
    return wrapper

def inner(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('Inner Before')
        result = func(*args, **kwargs)
        print('Inner After')
        return result
    return wrapper